# Task 1 Vast Train Notebook (`google/gemma-4-26B-A4B-it`)

Notebook nay dung cho luong `Vast.ai + RTX A6000 48GB`.

Thu tu chay:
1. install env
2. login Hugging Face
3. test model access
4. download data
5. dump prompt
6. smoke test
7. train that
8. upload checkpoint len Hugging Face Hub


In [ ]:
import os
from pathlib import Path

REPO_DIR = Path.cwd()
print("Repo dir:", REPO_DIR)
os.chdir(REPO_DIR)
%pip install -U pip setuptools wheel
%pip uninstall -y torch torchvision torchaudio bitsandbytes
%pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cu128 torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1
%pip install -r requirements.txt
%pip install --no-cache-dir bitsandbytes==0.47.0
%pip install -U git+https://github.com/huggingface/transformers.git


In [ ]:
import torch, transformers, huggingface_hub, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("peft:", peft.__version__)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader


In [ ]:
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN"
MODEL_ID = "google/gemma-4-26B-A4B-it"
RUN_NAME = "task1_gemma26b_a4b_it_vast"
HF_MODEL_REPO_ID = "SpringWang08/multimodal-empathy-mental-health-gemma26b-task1"

login(token=HF_TOKEN, add_to_git_credential=False)
print("HF login ok")


In [ ]:
from transformers import AutoProcessor, AutoTokenizer

processor = None
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    print("Processor loaded:", processor.__class__.__name__)
except Exception as e:
    print("AutoProcessor unavailable in current environment:", repr(e))
    print("Falling back to tokenizer-only path for text-only SFT.")

tokenizer = getattr(processor, "tokenizer", None)
if tokenizer is None:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


## Nếu bạn gặp lỗi `AttributeError: 'list' object has no attribute 'keys'`

Đó thường là do `transformers` trong môi trường chưa đủ mới để đọc tokenizer config của Gemma 4. Cell install đầu notebook đã cài `transformers` trực tiếp từ GitHub để tránh lỗi này.

Nếu bạn vừa sửa môi trường, hãy **restart kernel** rồi chạy lại từ đầu.


In [ ]:
!bash scripts/download_avamerg.sh
!bash scripts/download_esconv.sh
!mkdir -p outputs/sft outputs/eval

from pathlib import Path

ava_path = Path("data/raw/avamerg/train.json")
esc_path = Path("data/raw/esconv/ESConv.json")
print("AvaMERG file exists:", ava_path.exists(), ava_path)
print("ESConv file exists:", esc_path.exists(), esc_path)
if not ava_path.exists() or not esc_path.exists():
    raise FileNotFoundError(
        "Dataset download chua xong hoac bi fail. Kiem tra output cua cell download, HF login, va xem file co nam trong data/raw/... khong."
    )


In [ ]:
import json
from pathlib import Path

ava_path = Path("data/raw/avamerg/train.json")
esc_path = Path("data/raw/esconv/ESConv.json")
if not ava_path.exists():
    raise FileNotFoundError(f"Missing {ava_path}. Hay chay lai cell download phia tren va dam bao da `hf auth login`.")
if not esc_path.exists():
    raise FileNotFoundError(f"Missing {esc_path}. Hay chay lai cell download phia tren.")

ava = json.loads(ava_path.read_text(encoding="utf-8"))
esc = json.loads(esc_path.read_text(encoding="utf-8"))
print("AvaMERG samples:", len(ava))
print("ESConv dialogues:", len(esc))
print("AvaMERG first keys:", list(ava[0].keys()))
print("ESConv first keys:", list(esc[0].keys()))


In [ ]:
from argparse import Namespace
from scripts.train_sft import run_training

base_cfg = dict(
    model_name_or_path=MODEL_ID,
    avamerg_root="data/raw/avamerg",
    avamerg_split="train",
    avamerg_text_only=True,
    esconv_json="data/raw/esconv/ESConv.json",
    output_dir="outputs/sft/debug_gemma26b_a4b",
    max_length=1024,
    max_response_tokens=192,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=1.0,
    logging_steps=10,
    save_steps=100,
    warmup_ratio=0.03,
    max_steps=-1,
    use_lora=True,
    load_in_4bit=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    report_to="none",
    dump_example_prompts=True,
    max_train_samples=8,
    gradient_checkpointing=True,
)

run_training(Namespace(**base_cfg))


In [ ]:
!sed -n '1,200p' outputs/sft/debug_gemma26b_a4b/example_prompts.json


In [ ]:
smoke_cfg = dict(base_cfg)
smoke_cfg.update({
    "output_dir": "outputs/sft/gemma26b_a4b_smoke",
    "dump_example_prompts": False,
    "max_train_samples": 16,
    "gradient_accumulation_steps": 1,
    "max_steps": 1,
    "logging_steps": 1,
})

run_training(Namespace(**smoke_cfg))


In [ ]:
# Chi chay cell nay sau khi smoke test da qua.
train_cfg = dict(base_cfg)
train_cfg.update({
    "output_dir": f"outputs/sft/{RUN_NAME}",
    "dump_example_prompts": False,
    "max_train_samples": None,
    "gradient_accumulation_steps": 8,
    "logging_steps": 10,
    "save_steps": 100,
    "max_steps": -1,
})

run_training(Namespace(**train_cfg))


In [ ]:
from pathlib import Path

final_dir = Path(f"outputs/sft/{RUN_NAME}/final")
print("Final checkpoint dir:", final_dir)
print("Exists:", final_dir.exists())
if final_dir.exists():
    for path in sorted(final_dir.iterdir()):
        print(path.name)


In [ ]:
from scripts.publish_to_hub import create_repo, upload_folder

create_repo(repo_id=HF_MODEL_REPO_ID, repo_type="model", private=False, exist_ok=True)
upload_folder(
    repo_id=HF_MODEL_REPO_ID,
    repo_type="model",
    folder_path=str(final_dir),
    commit_message=f"Upload {RUN_NAME} checkpoint",
)
print(f"Uploaded {final_dir} -> https://huggingface.co/{HF_MODEL_REPO_ID}")


## Sau khi upload xong

- theo doi repo model tren Hugging Face
- neu khong dung may nua, stop hoac destroy instance tu Vast console
- checkpoint local van nam trong `outputs/sft/<run_name>/final`
